In [8]:
import re
import requests
from bs4 import BeautifulSoup
import ollama

# -----------------------------
# Local Gemma3 via Ollama
# gemma3:270m is too small for reliable translation (it echoes English).
# Use gemma3:1b or larger for multilingual tasks.
# -----------------------------
model_name = "gemma3:1b"

# -----------------------------
# Function: Extract website text
# -----------------------------
def extract_text_from_url(url):
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.extract()

    text = soup.get_text(separator=" ")
    clean_text = " ".join(text.split())
    return clean_text[:500]   # keep within LLM context window

# -----------------------------
# Function: Translate using LLM
# -----------------------------
def chunk_text(text, max_chars=400):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current = [], ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= max_chars:
            current = f"{current} {sentence}".strip()
        else:
            if current:
                chunks.append(current)
            current = sentence[:max_chars]

    if current:
        chunks.append(current)

    return chunks or [text[:max_chars]]


def translate_text(text, target_language):
    system_prompt = (
        f"You are a professional translator. Translate the user's text into {target_language}. "
        "Output ONLY the translation in the target language. "
        "Do not repeat the source text and do not add explanations."
    )

    translated_chunks = []
    for chunk in chunk_text(text):
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": chunk},
            ],
        )
        translated_chunks.append(response["message"]["content"].strip())

    return "\n\n".join(translated_chunks)

# -----------------------------
# Main usage
# -----------------------------
if __name__ == "__main__":
    url = "https://www.nebula.io/"   # put any website here
    target_language = "Telugu"    # change to any language

    website_text = extract_text_from_url(url)
    translated = translate_text(website_text, target_language)

    print("\n=== TRANSLATED OUTPUT ===\n")
    print(translated)



=== TRANSLATED OUTPUT ===

భీమియమైన శ్రామికనౌద్యులను మరియు కృత్రిమ మేధస్సుతో కూడిన నియమల కార్యక్రమం మరియు పంపిణీల వేదిక Nebula – ఉత్పత్తి పరిష్కార సృష్టి సంస్థ, Pricing Company ఆవరణకు వస్తుంది. </b>

నభున్ (Nebula) – ఉత్తమ అగ్రభావం కలిగిన సెలెక్షన్ క్రియల్‌ను సృష్టిస్తుంది, ఇది వినియోగదారులకు ప్రతిభ్యుత్తమి వస్తువులను విస్తరించడం, శీఘ్ర ఉద్యోగి ఎంపికలను చేయడంలో సహాయపడుతుంది మరియు వ్యూహాత్మక జనాభా అంతర్దృష్టులను అందించడంలో ఒకే విధమైన లొకేషన్‌కు మార్గనిర్దేశం చేస్తుంది. నభున్ ఒక నిర్దిష్ట సంఖ్యలో 200 M+ కౌట్‌అఫ్‌ను బుక్ చేయడం ద్వారా అభ్యర్థుల కోసం మన్‌స్టర్‌ను అప్‌డేట్ చేసేందుకు అందుబాటులో ఉంటుంది.
